This part of the pipeline processes the GTDB classification for the entire *Clostridia* genome set using the GTDB-Tk.

The instance hosted at the [European Galaxy server](usegalaxy.eu) was used to generate the raw classification data.

### Importing libraries

In [ ]:
import os
from os.path import join
import shutil
import pandas as pd
import matplotlib

### Paths and parameters

#### Pipeline input folders

In [ ]:
genomes = "01-bakta/genomes"
path_index_file = "01-bakta/all.list"
all_list = "01-bakta/all"

#### Pipeline output folders

In [ ]:
task_root = "02-GTDB"
galaxy_root = join(task_root, 'gtdbtk_galaxy')
subgroups = join(task_root, 'subgroups')

os.makedirs(task_root, exist_ok = True)
os.makedirs(galaxy_root, exist_ok = True)
os.makedirs(subgroups, exist_ok = True)

### Making a filename number index

In [ ]:
renamed_genomes_folder = join(task_root, "genomes_fasta")
os.makedirs(renamed_genomes_folder, exist_ok = True)

In [ ]:
genomes_file_index = pd.Series(sorted(os.listdir(genomes)))

#### Make a renamed copy

In [ ]:
for index, genome_file in enumerate(os.listdir(genomes)):
    shutil.copy(join(genomes, genome_file), join(renamed_genomes_folder, str(index) + '.fasta'))
shutil.make_archive(join(galaxy_root, 'genomes_fasta'), "zip", renamed_genomes_folder)

In [ ]:
shutil.rmtree(renamed_genomes_folder)

Upload the zip at Galaxy.eu and decompress it in the cloud. Then, run the full `classify_wf` workflow command of GTDB-Tk in the Galaxy cloud. Download all output and process it locally using the code below.

### Build classification table

In [ ]:
!tail -n +2 $galaxy_root/summary/gtdbtk.bac120.summary.tsv | cut -f 1,2 | tr ';' '\t' > $galaxy_root/summary/gtdb_classification

In [ ]:
table_path = join(galaxy_root, 'summary', 'gtdb_classification')
index_path = join(task_root, 'index')

In [ ]:
def deprefix(name):
    return name[3:]

def desuffix(name):
    return name[:-4]

levels = ['domain', 'phylum', 'class', 'order', 'family', 'genus', 'species']

In [ ]:
deprefix_levels = {i: deprefix for i in levels}

table = pd.read_table(table_path, sep = "\t", index_col = 0, header = None, names = levels, converters = deprefix_levels).sort_index()
index = pd.read_table(index_path, index_col= 0, header = None, names = ['accession'], converters = {'accession': desuffix})

classification = pd.merge(table, index, left_index = True, right_index = True).set_index('accession')
classification.to_csv(join(task_root, 'classification_table'), sep = "\t")

### Define grouping

Make filtered file indices in parallel as well.

In [ ]:
size_threshold = 30
group_level = "order"

In [ ]:
path_index = pd.read_table(path_index_file, sep = "\t", header = None, names = ['accession', 'path'])

In [ ]:
group_assignments = classification[group_level].to_frame()
group_level_uniques = classification[group_level].value_counts()
group_level_uniques = group_level_uniques[group_level_uniques >= size_threshold]
group_assignments = group_assignments[group_assignments[group_level].isin(group_level_uniques.index)]
group_assignments.to_csv(join(task_root, 'filtered_classification_table'), sep = "\t", index = True, header = False)

In [ ]:
group_assignments

In [ ]:
for group_name in group_assignments[group_level].unique():
    group = group_assignments[group_assignments[group_level] == group_name].reset_index()
    group.to_csv(join(subgroups, group_name), columns = ['accession'], index = False, header = False)

    group_path_index = path_index[path_index['accession'].isin(group['accession'])]
    group_path_index.to_csv(join(subgroups, group_name + '.list'), sep = '\t', index = False, header = False)

In [ ]:
shutil.copyfile(path_index_file, join(subgroups, 'all.list'))
shutil.copyfile(all_list, join(subgroups, 'all'))

### Assign iTol colours

In [ ]:
n_colors = len(group_level_uniques)
cmap = matplotlib.colormaps.get_cmap('tab20')
colour_index = range(n_colors)
colours = [matplotlib.colors.to_hex(cmap(i)) for i in colour_index]
legend = dict(zip(group_level_uniques.index, colours))

In [ ]:
legend_map = group_assignments.copy(deep = True)

In [ ]:
colour_match = []
for i in legend_map[group_level].to_list():
    try:      
        colour_match.append(legend[i])
    except KeyError:
        colour_match.append('')

In [ ]:
legend_map.insert(loc = 0, column = "colour", value = colour_match)
legend_map = legend_map[legend_map['colour'] != '']
legend_map.to_csv(join(task_root, 'iTol_colour_strip_export'), sep = "\t", index = True, header = False)